# 28d · CXR-LT — cứu `view_group` cho 15,769 ảnh `unknown`

`28_cxrlt_inventory.ipynb` gán `view_group='unknown'` cho **15,769 ảnh** thiếu `ViewPosition`, và
**9,695 study (4.26%) không còn ảnh frontal nào**. View pooling `0.7·frontal + 0.3·lateral` không
chạy được trên nhóm đó nếu không cứu.

Đã kiểm: `ViewCodeSequence_CodeMeaning` **cũng rỗng trên đúng 15,769 ảnh ấy** — hai trường DICOM
mất cùng nhau, nên metadata trong CSV của CXR-LT không cứu được gì.

Nhưng chỗ nào có cả hai thì ánh xạ gần như tất định, nên nếu tìm được nguồn khác thì tin được:

```
antero-posterior -> AP       146,442        lateral      -> LATERAL   82,612
postero-anterior -> PA        95,855        left lateral -> LL        35,033
```

Notebook leo thang từ rẻ đến đắt, **dừng ngay khi đủ**:

| | nguồn | chi phí |
|---|---|---|
| `C1` | `cohort_v2/view_map.parquet` — `03` đã giải rồi cho phần trùng cohort cũ | vài giây, **miễn phí** |
| `C2` | `mimic_cxr_jpg.metadata` trên BigQuery — các trường DICOM khác mà CXR-LT không chép | ~1 phút |
| `C3` | loại trừ trong cùng study | vài giây |
| `C4` | **vẽ ảnh** phần còn lại để mắt người phán | vài phút |

**CPU + BigQuery, không cần GPU.**

> `03` có một nguyên tắc phải giữ: **gán sai còn tệ hơn bỏ trống**, vì `view_group` chính là biến
> mà trọng số 7:3 dựa vào. Nên mọi bước dưới đây chỉ gán khi nguồn chứng minh được là tất định.

## C0 · Config

In [ ]:
import os, io, json, tarfile
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR  = os.path.join(DATA_DIR, 'cxrlt_2024')
OLD_COH  = os.path.join(DATA_DIR, 'cohort_v2')
SH_OLD   = os.path.join(DATA_DIR, 'cxr_560_shards')
SH_NEW   = os.path.join(DATA_DIR, 'cxr_560_shards_cxrlt')
P_IDX    = os.path.join(OUT_DIR, 'cxrlt_index.parquet')
assert os.path.exists(P_IDX), f'khong thay {P_IDX} -> chay 28_cxrlt_inventory.ipynb truoc'

PROJECT = 'mimic-ehr-v310'
CXR     = 'physionet-data.mimic_cxr_jpg'
PURITY_MIN = 0.98        # giong 03: mot ma chi duoc dung neu no tro ve MOT view voi do thuan nay

IDX = pd.read_parquet(P_IDX, columns=['dicom_id', 'study_id', 'subject_id',
                                      'split', 'view_group', 'ViewPosition'])
UNK = IDX[IDX.view_group == 'unknown'].copy()
print(f'{len(IDX):,} anh | unknown {len(UNK):,} ({len(UNK)/len(IDX):.2%})')
_nf = set(IDX.loc[IDX.view_group == 'frontal', 'study_id'])
print(f'study khong co frontal: {IDX.study_id.nunique() - len(_nf):,}')
RESOLVED = {}            # dicom_id -> 'frontal'|'lateral',  gom dan qua C1..C4

## C0-bis · Đăng nhập BigQuery bằng `gcloud`

`auth.authenticate_user()` không dùng được ở đây — tài khoản không phải tài khoản Cloud. Dùng
luồng device-flow: chạy cell dưới, mở link nó in ra, rồi **dán mã trả về ngược lại ô nhập**.

Phải làm lại **mỗi phiên Colab** — ADC không sống qua lần khởi động lại runtime.

In [ ]:
!gcloud auth application-default login --no-launch-browser

### C0-ter · Gán project rồi thử một query nhỏ

In [ ]:
!gcloud config set project {PROJECT}
!gcloud auth application-default set-quota-project {PROJECT} 2>/dev/null || true

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location='US')
print('BigQuery OK ->', [dict(r) for r in client.query('SELECT 1 AS ok').result()])

## C1 · Miễn phí — `03` đã giải phần nào rồi?

`03_view_resolution.ipynb` đã chạy cho `cohort_v2` và ghi `view_map.parquet` ở **mức ảnh**. Phần
CXR-LT trùng cohort cũ thì đáp án đã nằm sẵn ở đó, không phải tính lại.

In [ ]:
VM = os.path.join(OLD_COH, 'view_map.parquet')
if os.path.exists(VM):
    vm = pd.read_parquet(VM)
    print(f'view_map.parquet: {len(vm):,} dong | cot {list(vm.columns)}')
    if 'dicom_id' in vm.columns and 'view_group' in vm.columns:
        m = vm[['dicom_id', 'view_group']].drop_duplicates('dicom_id')
        hit = UNK[['dicom_id']].merge(m, on='dicom_id', how='inner')
        hit = hit[hit.view_group.isin(['frontal', 'lateral'])]
        RESOLVED.update(dict(zip(hit.dicom_id, hit.view_group)))
        print(f'\n03 da giai duoc: {len(hit):,} / {len(UNK):,} anh unknown')
        print(hit.view_group.value_counts().to_string())
    else:
        print('\n!! view_map khong o muc anh (thieu dicom_id/view_group) -> bo qua C1')
        print(f'   cot co: {list(vm.columns)}')
else:
    print(f'khong thay {VM} -> bo qua C1')

print(f'\nCON LAI: {len(UNK) - len(RESOLVED):,} anh')

## C2 · BigQuery — các trường DICOM mà CXR-LT **không chép vào CSV**

`labels.csv` chỉ mang `ViewPosition` + `ViewCodeSequence_CodeMeaning`, và cả hai rỗng trên nhóm
này. Nhưng bảng `mimic_cxr_jpg.metadata` gốc còn nhiều trường khác —
`PatientOrientationCodeSequence_CodeMeaning` là nhân chứng thứ hai mà `03` dùng.

Cell này **liệt kê cột thật của bảng** rồi mới quyết dùng cột nào, thay vì đoán tên.

In [ ]:
meta_cols = [f.name for f in client.get_table(f'{CXR}.metadata').schema]
CAND = [c for c in meta_cols
        if any(k in c.lower() for k in ('view', 'orientation', 'procedure', 'series', 'protocol'))]
print(f'cot lien quan den view trong {CXR}.metadata:')
for c in CAND: print(f'   {c}')

rest = [d for d in UNK.dicom_id if d not in RESOLVED]
print(f'\nhoi metadata cho {len(rest):,} anh con lai...')

_sel = ', '.join(CAND)
_lit = ',\n'.join(f"'{d}'" for d in rest[:200000])
q = f'''
SELECT dicom_id, {_sel}
FROM `{CXR}.metadata`
WHERE dicom_id IN ({_lit})
'''
M = client.query(q).to_dataframe()
print(f've {len(M):,} dong\n')
print('do phu tung cot tren nhom unknown:')
for c in CAND:
    nn = M[c].notna().sum() if c in M.columns else 0
    print(f'   {c:<52} {nn:>7,} ({nn/max(len(M),1):>6.1%})')

## C2-bis · Cổng — cột nào **tất định** đủ để gán?

Cùng luật của `03`: một mã chỉ được dùng nếu trên **những ảnh có nhãn** nó trỏ về đúng một
`view_group` với độ thuần ≥ `PURITY_MIN`. Mã nào chia đôi là mã mơ hồ, **không được dùng**.

In [ ]:
FRONTAL = {'AP', 'PA', 'AP AXIAL', 'PA LLD', 'PA RLD', 'AP LLD', 'AP RLD'}
LATERAL = {'LATERAL', 'LL', 'RL', 'LAO', 'RAO', 'LPO', 'RPO', 'SWIMMERS', 'XTABLE LATERAL'}
_vp = IDX.ViewPosition.fillna('').str.upper().str.strip()
LABELLED = IDX.assign(vg=np.select([_vp.isin(FRONTAL), _vp.isin(LATERAL)],
                                   ['frontal', 'lateral'], default=None))
LABELLED = LABELLED[LABELLED.vg.notna()][['dicom_id', 'vg']]

USABLE = {}
for c in [c for c in CAND if c in M.columns and M[c].notna().any()]:
    ref = client.query(f'''
        SELECT dicom_id, {c} AS code FROM `{CXR}.metadata` WHERE {c} IS NOT NULL
    ''').to_dataframe().merge(LABELLED, on='dicom_id')
    if not len(ref): continue
    t = ref.groupby(['code', 'vg']).size().unstack(fill_value=0)
    pur = t.max(axis=1) / t.sum(axis=1)
    good = t.index[(pur >= PURITY_MIN) & (t.sum(axis=1) >= 50)]
    if len(good):
        USABLE[c] = {k: t.loc[k].idxmax() for k in good}
        print(f'\n{c}: {len(good)}/{len(t)} ma dat do thuan >= {PURITY_MIN}')
        print(t.loc[good].assign(do_thuan=pur[good].round(4)).to_string())
    else:
        print(f'\n{c}: KHONG ma nao dat do thuan -> khong dung')

# --- ap dung ---
n0 = len(RESOLVED)
for c, mp in USABLE.items():
    if c not in M.columns: continue
    for r in M[['dicom_id', c]].dropna().itertuples(index=False):
        if r[0] not in RESOLVED and r[1] in mp:
            RESOLVED[r[0]] = mp[r[1]]
print(f'\nC2 giai them: {len(RESOLVED)-n0:,} anh   |   CON LAI: {len(UNK)-len(RESOLVED):,}')

## C3 · Loại trừ trong cùng study

Phim ngực thường chụp một cặp **thẳng + nghiêng**. Nếu một study có đúng 2 ảnh, một cái đã biết là
frontal và cái kia chưa biết, thì cái chưa biết **rất có thể** là lateral.

`03` gọi đây là "within-study elimination". Nó là **suy luận**, không phải bằng chứng — nên cell
này chỉ *đề xuất*, ghi vào cột riêng, và **không** trộn vào `RESOLVED` cho tới khi C4 xác nhận
bằng mắt.

In [ ]:
rest = [d for d in UNK.dicom_id if d not in RESOLVED]
R = IDX[IDX.dicom_id.isin(rest)][['dicom_id', 'study_id']]

known = IDX[IDX.view_group.isin(['frontal', 'lateral'])]
known = known.assign(**{'vg': known.view_group})
per = (known.groupby('study_id').vg.agg(
          n_f=lambda s: (s == 'frontal').sum(), n_l=lambda s: (s == 'lateral').sum()))
n_unk = R.groupby('study_id').size().rename('n_unk')
ctx = R.merge(per, on='study_id', how='left').merge(n_unk, on='study_id', how='left')
ctx[['n_f', 'n_l']] = ctx[['n_f', 'n_l']].fillna(0).astype(int)

# de xuat: study co dung 1 anh unknown, da co frontal, chua co lateral -> doan lateral
ctx['de_xuat'] = np.where((ctx.n_unk == 1) & (ctx.n_f >= 1) & (ctx.n_l == 0), 'lateral',
                  np.where((ctx.n_unk == 1) & (ctx.n_l >= 1) & (ctx.n_f == 0), 'frontal', None))
print(f'{len(ctx):,} anh chua giai duoc, boi canh study:')
print(ctx.groupby(['n_f', 'n_l', 'n_unk']).size().rename('n_anh')
         .sort_values(ascending=False).head(12).to_string())
print(f'\nde xuat duoc: {ctx.de_xuat.notna().sum():,}')
print(ctx.de_xuat.value_counts(dropna=False).to_string())
print(f'\nMO COI (study chi co anh unknown, khong co gi de doi chieu): '
      f'{int(((ctx.n_f == 0) & (ctx.n_l == 0)).sum()):,}')

## C4 · Vẽ ảnh — để mắt người phán

Ba nhóm cạnh nhau. Hàng đầu là **tham chiếu** (ảnh đã biết chắc), để hiệu chỉnh mắt trước khi nhìn
nhóm chưa biết.

Phim **nghiêng** nhìn là ra ngay: thấy rõ cột sống chồng lên tim, xương ức phía trước, hai vòm
hoành xếp lớp. Phim **thẳng** thì lồng ngực đối xứng, hai xương đòn ngang nhau.

In [ ]:
import matplotlib.pyplot as plt

# --- ban do dicom_id -> (thu muc shard, ten tar) ---
LOC = {}
for d, pref in [(SH_OLD, 'shard_'), (SH_NEW, 'cxrlt_')]:
    wl = os.path.join(d, '_work_list.parquet')
    if not os.path.exists(d) or not os.path.exists(wl): continue
    w = pd.read_parquet(wl, columns=['dicom_id', 'shard'])
    for r in w.itertuples(index=False):
        LOC[r.dicom_id] = (d, f'{pref}{int(r.shard):05d}.tar')
print(f'ban do vi tri anh: {len(LOC):,} dicom_id')

_cache = {}
def load(dic):
    if dic not in LOC: return None
    d, tarn = LOC[dic]
    key = (d, tarn)
    if key not in _cache:
        with tarfile.open(os.path.join(d, tarn)) as t:
            _cache.clear()
            _cache[key] = {m.name[:-4]: t.extractfile(m).read()
                           for m in t.getmembers() if m.name.endswith('.jpg')}
    b = _cache[key].get(dic)
    return plt.imread(io.BytesIO(b), format='jpeg') if b else None

def grid(items, title, ncol=6):
    items = [(d, lb) for d, lb in items if d in LOC][:ncol]
    if not items:
        print(f'({title}: khong co anh nao doc duoc)'); return
    fig, ax = plt.subplots(1, len(items), figsize=(2.4*len(items), 2.9))
    ax = np.atleast_1d(ax)
    for a, (d, lb) in zip(ax, items):
        im = load(d)
        if im is None: a.axis('off'); continue
        a.imshow(im, cmap='gray'); a.set_title(lb, fontsize=8); a.axis('off')
    fig.suptitle(title, fontsize=11, y=1.06); plt.tight_layout(); plt.show()

rng = np.random.default_rng(0)
_ref = IDX[IDX.view_group.isin(['frontal', 'lateral'])]
grid([(r.dicom_id, f'THAM CHIEU\n{r.ViewPosition}')
      for r in _ref[_ref.view_group == 'frontal'].sample(6, random_state=1).itertuples()],
     'THAM CHIEU — frontal (da biet chac)')
grid([(r.dicom_id, f'THAM CHIEU\n{r.ViewPosition}')
      for r in _ref[_ref.view_group == 'lateral'].sample(6, random_state=1).itertuples()],
     'THAM CHIEU — lateral (da biet chac)')

_dx = ctx[ctx.de_xuat.notna()]
if len(_dx):
    grid([(r.dicom_id, f'de xuat: {r.de_xuat}\n(study co {r.n_f}F/{r.n_l}L)')
          for r in _dx.sample(min(6, len(_dx)), random_state=2).itertuples()],
         'UNKNOWN — co de xuat tu loai tru trong study  (kiem: co dung khong?)')

_or = ctx[(ctx.n_f == 0) & (ctx.n_l == 0)]
if len(_or):
    grid([(r.dicom_id, 'mo coi\n(khong doi chieu duoc)')
          for r in _or.sample(min(6, len(_or)), random_state=3).itertuples()],
         'UNKNOWN — mo coi, chi co the phan bang mat')

print(f'''
================================================================
TINH DEN GIO

  unknown ban dau        : {len(UNK):,}
  da giai (C1 + C2)      : {len(RESOLVED):,}
  co de xuat (C3)        : {int(ctx.de_xuat.notna().sum()):,}   <- nhin anh de quyet co nhan khong
  mo coi                 : {int(((ctx.n_f == 0) & (ctx.n_l == 0)).sum()):,}

Nhin hai hang cuoi roi dat ACCEPT_C3 o C5.
================================================================''')

## C5 · Chốt — cái gì cứu được thì giữ, còn lại **loại**

Chính sách: **không cứu được thì bỏ ảnh đó, không học**. Không chế quy tắc dự phòng, vì gán sai
`view_group` làm hỏng chính trọng số 7:3 mà việc gộp view dựa vào.

Nhưng phải đo một hệ quả mà "bỏ ảnh" nghe qua không lộ ra: **bỏ ảnh ≠ bỏ study**. Study nào mà
*toàn bộ* ảnh đều bị loại thì biến mất hẳn khỏi dataset. Nếu điều đó xảy ra ở `val`/`test` thì số
đo của ta không còn nằm trên split chính thức nữa — nên cell này đếm chính xác chuyện đó, tách
theo split.

Đặt `ACCEPT_C3 = True` nếu C4 cho thấy các đề xuất loại-trừ-trong-study là đúng.

In [ ]:
ACCEPT_C3 = True        # <<< dat False neu C4 cho thay de xuat khong dang tin

FINAL = IDX[['dicom_id', 'study_id', 'subject_id', 'split', 'view_group']].copy()
FINAL['nguon_view'] = np.where(FINAL.view_group == 'unknown', None, 'ViewPosition')

_r = pd.Series(RESOLVED)
if len(_r):
    m = FINAL.dicom_id.map(_r)
    FINAL.loc[m.notna(), 'view_group'] = m[m.notna()]
    FINAL.loc[m.notna(), 'nguon_view'] = 'C1/C2 metadata'
if ACCEPT_C3:
    _d = ctx.set_index('dicom_id').de_xuat.dropna()
    m = FINAL.dicom_id.map(_d)
    take = m.notna() & (FINAL.view_group == 'unknown')
    FINAL.loc[take, 'view_group'] = m[take]
    FINAL.loc[take, 'nguon_view'] = 'C3 loai tru trong study'

FINAL['use'] = FINAL.view_group.isin(['frontal', 'lateral'])
print('nguon cua view_group cuoi cung:')
print(FINAL.nguon_view.value_counts(dropna=False).to_string())
print(f'\nGIU  : {int(FINAL.use.sum()):,} anh')
print(f'LOAI : {int((~FINAL.use).sum()):,} anh  ({(~FINAL.use).mean():.2%})')

# ---- HE QUA THAT SU: bao nhieu STUDY bien mat ----
alive = set(FINAL.loc[FINAL.use, 'study_id'])
S0 = IDX.groupby('split').study_id.nunique()
S1 = FINAL[FINAL.use].groupby('split').study_id.nunique()
rep = pd.DataFrame(dict(study_truoc=S0, study_sau=S1)).fillna(0).astype(int)
rep['mat'] = rep.study_truoc - rep.study_sau
rep['ty_le_mat'] = (rep.mat / rep.study_truoc).map('{:.3%}'.format)
print('\n=== STUDY BIEN MAT (day moi la con so quan trong) ===')
print(rep.reindex(['train', 'val', 'test']).to_string())

_lost_eval = int(rep.loc[['val', 'test'], 'mat'].sum())
if _lost_eval:
    print(f'''
!! {_lost_eval:,} study cua VAL/TEST bi loai hoan toan.
   Hau qua: so do KHONG con nam tron tren split chinh thuc CXR-LT nua.
   Bat buoc ghi ro con so nay canh moi bang ket qua. Neu no lon, can can nhac
   giu lai nhung study do voi view_group='frontal' (gia dinh mac dinh) CHI cho
   viec cham diem, khong dua vao train.''')
else:
    print('\nOK  khong study VAL/TEST nao bi mat -> so do van tron tren split chinh thuc')

P = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
FINAL.to_parquet(P, index=False)
print(f'\nghi {P}')
print('   -> 31/32 doc file NAY de lay view_group va co `use`, '
      'KHONG doc lai cot view_group cua cxrlt_index.parquet')
print('   (cxrlt_index.parquet khong bi sua)')